# Voxel Head Model

This notebook introduces `VoxelHeadModel`, an alternative to
`TwoSurfaceHeadModel` that represents the brain as a reduced set of
**voxels** rather than a triangulated cortex surface.  Reconstruction
targets are then individual voxels distributed through the illuminated
volume rather than vertices on a cortical sheet.  See
`40_image_reconstruction.ipynb` for the full surface-based pipeline; this
notebook covers what changes when the brain is voxelised.

The pipeline mirrors the surface case:

1. Load a voxel head model from a standard atlas.
2. Snap optodes to the scalp; reduce voxels to those covered by the probe.
3. Run Monte-Carlo photon transport (or load precomputed fluence).
4. **Reduce voxels by fluence** — runs *between* MCX and
   `compute_sensitivity` so the resulting sensitivity matrix is small.
5. Compute the sensitivity matrix `Adot` directly on the kept voxels.
6. Drop remaining low-sensitivity voxels with `reduce_voxels_to_sensitivity`.

The resulting `Adot` carries the same dims and coords as the surface
variant's, so it is consumed by `ImageRecon` in exactly the same way; see
`40_image_reconstruction.ipynb` for the reconstruction step.

In [1]:
# This cell sets up the environment when executed in Google Colab.
try:
    import google.colab
    get_ipython().system('curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py')
    get_ipython().run_line_magic('run', 'colab_setup.py')
except ImportError:
    pass

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import trimesh
import xarray as xr
from matplotlib.collections import LineCollection
from scipy.spatial import KDTree
from trimesh.intersections import mesh_plane

import cedalion
import cedalion.data
import cedalion.dataclasses as cdc
import cedalion.dot as dot
import cedalion.dot.forward_model as fw
import cedalion.nirs
import cedalion.sim.synthetic_hrf as synhrf
from cedalion import units

xr.set_options(display_expand_data=False)


def as_mm(obj):
    """Positions of a surface, point cloud or landmark selection as (n, 3) in mm."""
    if hasattr(obj, "pint") and getattr(obj.pint, "units", None) is not None:
        obj = obj.pint.to("mm").pint.dequantify()
    return np.atleast_2d(np.asarray(obj, dtype=float))

## 1. Load a voxel head model

`get_standard_headmodel(model, kind="voxel")` returns a `VoxelHeadModel`
built from the same Colin27/ICBM152 segmentation masks that back the
surface variant.  The brain is meshed as voxels rather than as a
triangulated cortex; the scalp surface is reused from the atlas.

In [3]:
head = dot.get_standard_headmodel("colin27", kind="voxel")
print(head)

VoxelHeadModel(
  crs: ijk
  tissue_types: csf, gm, scalp, skull, wm
  brain voxels: 1607695 units: dimensionless
  scalp faces: 20096 vertices: 10050 units: dimensionless
  landmarks: 73
)


Compared to the surface variant, the brain attribute is now a
`cdc.Voxels` cloud rather than a mesh.  The sparse mapping
`voxel_to_vertex_brain` projects the full segmentation grid onto these
kept voxels — the columns are voxels here, not mesh vertices.  Code
that prefers a clearer name can use the alias
`voxel_to_voxel_brain`:

In [4]:
print("brain voxels:", head.brain.nvertices)
print("voxel_to_vertex_brain shape:", head.voxel_to_vertex_brain.shape)
print("alias property identity:", head.voxel_to_voxel_brain is head.voxel_to_vertex_brain)

brain voxels: 1607695
voxel_to_vertex_brain shape: (7109137, 1607695)
alias property identity: True


## 2. Probe-based reduction

Most montages cover only part of the head.  Brain voxels that no optode
can plausibly see can be dropped before any photon transport simulation.

In [5]:
rec = cedalion.data.get_fingertappingDOT()

# snap the recording's optode positions to the scalp surface
geo3d_snapped = head.align_and_snap_to_scalp(rec.geo3d)

# drop voxels far from any optode
head_ijk = head.reduce_voxels_to_probe(geo3d_snapped, max_dist=4 * units.cm)
print("after reduce_voxels_to_probe:", head_ijk.brain.nvertices, "voxels")

after reduce_voxels_to_probe: 921697 voxels


## 3. Use precomputed fluence

The full Monte-Carlo step (`ForwardModel.compute_fluence_mcx`) requires
an NVIDIA GPU and is omitted from the notebook output.  We use a
precomputed fluence HDF5 file instead.  Note the fluence file lives on
the *full* segmentation voxel grid — independent of which head model
(surface or voxel) consumes it.

In [6]:
fluence_fname = cedalion.data.get_precomputed_fluence("fingertappingDOT", "colin27")
print("fluence file:", Path(fluence_fname).name)

fluence file: fluence_fingertappingDOT_colin27.h5


## 4. Fluence reducer

`reduce_voxels_by_fluence` reads the fluence file, computes per-voxel
$\sum_\text{optodes} |\text{fluence}|$ (max over wavelengths) and drops
voxels below `rel_threshold * max`. Runs *before* `compute_sensitivity`, so the
resulting `Adot` is small from the start.

In [7]:
head_ijk = head_ijk.reduce_voxels_by_fluence(fluence_fname, rel_threshold=1e-3)
print("after reduce_voxels_by_fluence:", head_ijk.brain.nvertices, "voxels")

after reduce_voxels_by_fluence: 501734 voxels


## 5. Compute sensitivity

`ForwardModel.compute_sensitivity` works exactly the same on a voxel
head model — the duck-typed attribute names match.  The resulting
`Adot` has dims `(channel, vertex, wavelength)` where `vertex` indexes
the kept voxels.

In [8]:
measurement_list = rec._measurement_lists["amp"]

fwm = fw.ForwardModel(head_ijk, geo3d_snapped, measurement_list)

with TemporaryDirectory() as tmp_dir:
    sens_path = Path(tmp_dir) / "sensitivity.nc"
    fwm.compute_sensitivity(fluence_fname, sens_path)
    Adot = cedalion.io.forward_model.load_Adot(sens_path)

print(Adot.dims, Adot.shape)
print("is_brain True count:", int(Adot.is_brain.sum()))

('channel', 'vertex', 'wavelength') (100, 511784, 2)
is_brain True count: 501734


## 6. Drop the remaining low-sensitivity voxels

After `compute_sensitivity`, voxels whose summed absolute sensitivity
$\sum_\text{channel} \sum_\text{wavelength} |A_{ij}|$ stays below an
absolute threshold carry no usable information and are removed.  The
reducer returns the new head model and the matching `Adot` in lockstep.
Everything below runs on this reduced pair.

In [9]:
head_ijk_final, Adot_final = head_ijk.reduce_voxels_to_sensitivity(
    Adot, sensitivity_threshold=1e-4
)
print("after reduce_voxels_to_sensitivity:", head_ijk_final.brain.nvertices, "voxels")

after reduce_voxels_to_sensitivity: 238723 voxels
